# HIMYM `latest_predictions.csv` False Positive Analysis

This notebook explores [`manual_tests/datasets/himym/s04e12_benefits_l1055_1259/latest_predictions.csv`](../manual_tests/datasets/himym/s04e12_benefits_l1055_1259/latest_predictions.csv).

Goal:
- explain why the run produces so many false positives
- identify repeatable patterns
- group every FP into a concrete root-cause bucket
- recommend what to change for each bucket

High-level finding before we drill down: the problem is mostly **idiom over-generation**, not phrasal verbs. Out of 60 false positives, **55 come from `engine_idiom`** and only **5 come from `engine_phrasal_verb`**.


In [1]:
from pathlib import Path
import csv
from collections import Counter, defaultdict
import textwrap

DATASET_DIR = Path("../manual_tests/datasets/himym/s04e12_benefits_l1055_1259")
PRED_PATH = DATASET_DIR / "latest_predictions.csv"
GOLD_PATH = DATASET_DIR / "gold_v.1.0.csv"

def load_csv(path: Path):
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def pct(part, whole):
    return 0.0 if whole == 0 else round(part * 100 / whole, 1)

def clip(text, width=110):
    return textwrap.shorten(" ".join(text.split()), width=width, placeholder="...")

rows = load_csv(PRED_PATH)
gold_rows = load_csv(GOLD_PATH)

fps = [r for r in rows if r["match_status"] == "fp"]
tps = [r for r in rows if r["match_status"] == "tp"]
fns = [r for r in rows if r["match_status"] == "fn"]

print(f"Prediction rows: {len(rows)}")
print(f"Gold rows:       {len(gold_rows)}")
print(f"TP / FP / FN:    {len(tps)} / {len(fps)} / {len(fns)}")
print()
print("False positives by detector:")
for detector, count in Counter(r["prediction_detector"] for r in fps).most_common():
    print(f"  {detector:20} {count:>2} ({pct(count, len(fps)):>4}%)")
print()
print("False positives by predicted type:")
for expr_type, count in Counter(r["prediction_type"] for r in fps).most_common():
    print(f"  {expr_type:20} {count:>2} ({pct(count, len(fps)):>4}%)")


Prediction rows: 89
Gold rows:       29
TP / FP / FN:    16 / 60 / 13

False positives by detector:
  engine_idiom         55 (91.7%)
  engine_phrasal_verb   5 ( 8.3%)

False positives by predicted type:
  idiom                55 (91.7%)
  phrasal_verb          5 ( 8.3%)


In [2]:
print("Top FP expressions:")
for expr, count in Counter(r["prediction_expression"] for r in fps).most_common(20):
    print(f"  {count:>2}  {expr}")

print("\nLines with the most FPs:")
by_line = defaultdict(list)
for row in fps:
    by_line[int(row["line_index"])] += [row]

for line_index, items in sorted(by_line.items(), key=lambda kv: (-len(kv[1]), kv[0])):
    print(f"  line {line_index}: {len(items)} FP(s) | {clip(items[0]['line'], 120)}")

print("\nFP lines in detail:")
for line_index in sorted(by_line):
    items = by_line[line_index]
    print(f"\nline {line_index} | {items[0]['speaker']}")
    print(f"  text: {clip(items[0]['line'], 150)}")
    for item in items:
        print(f"  - {item['prediction_detector']} | {item['prediction_type']} | {item['prediction_expression']}")


Top FP expressions:
   2  get out of here
   2  get something out
   2  take someone in
   2  take someone or an animal in
   2  take someone something or an animal inside
   2  take something in
   1  take someone or something out
   1  take someone out
   1  take something out
   1  couple something together
   1  go there
   1  no go
   1  what happened
   1  boil down to something
   1  boil something down
   1  boil down to
   1  do what
   1  in there
   1  read something in something
   1  at work

Lines with the most FPs:
  line 1255: 9 FP(s) | Right. It's not like you, you know? In addition, we are friends. I want to complicate matters by committing. Hanging...
  line 1209: 8 FP(s) | Because of your bickering roommates are always a source of conflict between you two, I wanted to help. In fact, I...
  line 1164: 5 FP(s) | No, it's wrong. You must learn to get it out. As we did in my kindergarten class. "The time for emotions", every...
  line 1211: 5 FP(s) | I took it in passin

## Root-cause buckets

The buckets below are manual labels over the existing FP rows. They are meant to be operational, not philosophical: each group should suggest a fix.

1. `templatic_placeholder_variants`
   Idiom lexicon entries containing `someone`, `something`, `some place`, etc. are matching ordinary literal syntax and often produce several FPs for one surface span.

2. `syntax_blind_matcher`
   The matcher only checks lemma sequences with a small gap allowance. It ignores punctuation, clause boundaries, POS, and dependency structure, so accidental matches survive.

3. `inventory_too_broad`
   Some lexicon entries are simply too generic for default idiom detection: `at work`, `more and more`, `go there`, `do what`.

4. `duplicate_on_true_mwe`
   One real or near-real MWE occurrence triggers several related lexicon entries around the same span, e.g. `take out`, `boil down`, `get out`, `hang out`.

5. `pv_or_gold_scope`
   Small residue caused by phrasal-verb inventory issues or annotation granularity mismatches.

Code paths worth keeping in mind while reading the output:
- [`manual_tests/run_mwe_eval.py`](../manual_tests/run_mwe_eval.py): idiom candidate generation and evaluation loop
- [`app/search/mwe/idioms.py`](../app/search/mwe/idioms.py): `_find_lemma_sequence()` uses syntax-free lemma matching with up to a 2-token gap
- [`app/search/mwe/phrasal_verbs.py`](../app/search/mwe/phrasal_verbs.py): phrasal verb matching and PV filters


In [3]:
CATEGORY_INFO = {
    "templatic_placeholder_variants": {
        "label": "Placeholder variants",
        "why": "Lexicon patterns with someone/something slots are matching literal syntax and spawning multiple variants.",
        "what_to_do": "Down-rank or disable slot-heavy idiom patterns by default, and dedupe predictions by matched span before scoring/display.",
    },
    "syntax_blind_matcher": {
        "label": "Syntax-blind matching",
        "why": "Lemma sequence matching ignores punctuation, part of speech, clause boundaries, and dependency structure.",
        "what_to_do": "Add sentence-boundary and POS/dependency constraints; require tighter local structure for idiom matches.",
    },
    "inventory_too_broad": {
        "label": "Inventory too broad",
        "why": "Some default idiom entries are common compositional or discourse phrases and should not fire freely.",
        "what_to_do": "Create an allowlist/tiered lexicon and demote generic entries from default detection.",
    },
    "duplicate_on_true_mwe": {
        "label": "Duplicate around true MWE",
        "why": "A true or near-true MWE span is causing several neighboring canonical entries to fire.",
        "what_to_do": "Cluster predictions by token span and keep the best canonical form instead of returning every neighboring variant.",
    },
    "pv_or_gold_scope": {
        "label": "PV or gold-scope mismatch",
        "why": "A small set comes from phrasal-verb inventory choices or annotation granularity mismatches.",
        "what_to_do": "Review these manually; either tighten the PV inventory or expand gold guidelines where the expression is arguably valid.",
    },
}

CATEGORY_MAP = {
    "duplicate_on_true_mwe": {
        (1061, "take someone or something out"),
        (1061, "take someone out"),
        (1061, "take something out"),
        (1072, "boil down to something"),
        (1072, "boil something down"),
        (1072, "boil down to"),
        (1110, "end in something"),
        (1116, "spice something up"),
        (1164, "get it out"),
        (1164, "get something out"),
        (1169, "lay around"),
        (1169, "leave something lying around"),
        (1169, "lie around some place"),
        (1191, "all for the best"),
        (1229, "get out of here"),
        (1229, "get something out"),
        (1229, "get out!"),
    },
    "templatic_placeholder_variants": {
        (1093, "room with someone"),
        (1178, "mean nothing to someone"),
        (1209, "help someone in something"),
        (1209, "help someone into something"),
        (1209, "take someone in"),
        (1209, "take someone or an animal in"),
        (1209, "take someone something or an animal inside"),
        (1209, "take something in"),
        (1209, "want someone or something in something"),
        (1211, "take someone in"),
        (1211, "take someone or an animal in"),
        (1211, "take someone something or an animal inside"),
        (1211, "take something in"),
        (1241, "have at someone"),
        (1242, "get someone going"),
        (1255, "hang out of something"),
        (1255, "hang out some place"),
        (1255, "hang out with someone or something"),
        (1255, "hang someone or something with something"),
        (1255, "hang something out of something"),
        (1255, "hang with someone"),
    },
    "syntax_blind_matcher": {
        (1068, "couple something together"),
        (1068, "no go"),
        (1068, "what happened"),
        (1091, "in there"),
        (1091, "read something in something"),
        (1093, "that there"),
        (1116, "used to someone or something"),
        (1164, "done in"),
        (1191, "for good"),
        (1209, "want into something"),
        (1211, "take in"),
        (1242, "ive got to go"),
        (1255, "like you know"),
    },
    "inventory_too_broad": {
        (1068, "go there"),
        (1087, "do what"),
        (1093, "at work"),
        (1164, "time for someone or something"),
        (1204, "more and more"),
        (1255, "in addition to something"),
    },
    "pv_or_gold_scope": {
        (1164, "get out of here"),
        (1241, "do better"),
        (1255, "hang with"),
    },
}

reverse_category_map = {}
for category, items in CATEGORY_MAP.items():
    for key in items:
        reverse_category_map[key] = category

fp_keys = {(int(r["line_index"]), r["prediction_expression"]) for r in fps}
assert fp_keys == set(reverse_category_map), (
    f"FP coverage mismatch. missing={sorted(fp_keys - set(reverse_category_map))} extra={sorted(set(reverse_category_map) - fp_keys)}"
)

labeled_fps = []
for row in fps:
    key = (int(row["line_index"]), row["prediction_expression"])
    labeled_fps.append({
        **row,
        "category": reverse_category_map[key],
        "line_index_int": int(row["line_index"]),
    })

summary = []
for category, info in CATEGORY_INFO.items():
    items = [r for r in labeled_fps if r["category"] == category]
    summary.append((len(items), category, info["label"], info["what_to_do"]))

for count, category, label, action in sorted(summary, reverse=True):
    print(f"{label:28} {count:>2} FP(s)  {pct(count, len(labeled_fps)):>4}%")
    print(f"  Fix: {action}")


Placeholder variants         21 FP(s)  35.0%
  Fix: Down-rank or disable slot-heavy idiom patterns by default, and dedupe predictions by matched span before scoring/display.
Duplicate around true MWE    17 FP(s)  28.3%
  Fix: Cluster predictions by token span and keep the best canonical form instead of returning every neighboring variant.
Syntax-blind matching        13 FP(s)  21.7%
  Fix: Add sentence-boundary and POS/dependency constraints; require tighter local structure for idiom matches.
Inventory too broad           6 FP(s)  10.0%
  Fix: Create an allowlist/tiered lexicon and demote generic entries from default detection.
PV or gold-scope mismatch     3 FP(s)   5.0%
  Fix: Review these manually; either tighten the PV inventory or expand gold guidelines where the expression is arguably valid.


In [4]:
for category, info in sorted(CATEGORY_INFO.items(), key=lambda kv: -len([r for r in labeled_fps if r['category'] == kv[0]])):
    items = sorted(
        [r for r in labeled_fps if r["category"] == category],
        key=lambda r: (r["line_index_int"], r["prediction_expression"]),
    )
    print(f"\n=== {info['label']} ({len(items)} FP / {pct(len(items), len(labeled_fps))}%) ===")
    print(f"Why it happens: {info['why']}")
    print(f"What to do:     {info['what_to_do']}")
    print("Examples:")
    for row in items[:8]:
        print(
            f"  line {row['line_index']} | {row['prediction_expression']} | {clip(row['line'], 115)}"
        )



=== Placeholder variants (21 FP / 35.0%) ===
Why it happens: Lexicon patterns with someone/something slots are matching literal syntax and spawning multiple variants.
What to do:     Down-rank or disable slot-heavy idiom patterns by default, and dedupe predictions by matched span before scoring/display.
Examples:
  line 1093 | room with someone | If this is a problem. You've done all the way here to read a magazine? I am willing to bet that there is a place...
  line 1178 | mean nothing to someone | No. It meant nothing. It was just a reflex when we were a couple. But I did everything go wrong.
  line 1209 | help someone in something | Because of your bickering roommates are always a source of conflict between you two, I wanted to help. In fact,...
  line 1209 | help someone into something | Because of your bickering roommates are always a source of conflict between you two, I wanted to help. In fact,...
  line 1209 | take someone in | Because of your bickering roommates are always a 

## Interpretation

The numbers point to one dominant conclusion: the false positives are not primarily caused by the phrasal-verb detector. They are mostly caused by the idiom pipeline in [`manual_tests/run_mwe_eval.py`](../manual_tests/run_mwe_eval.py), which builds candidates from the full McGraw-Hill pattern inventory, and by [`app/search/mwe/idioms.py`](../app/search/mwe/idioms.py), which accepts syntax-free lemma matches.

The most actionable pattern is the combination of:
- broad placeholder-rich idiom entries
- syntax-blind matching
- no span-level deduplication when several nearby canonical entries map to the same local phrase

That combination is what creates lines like `1209`, `1211`, and `1255`, where a single local context explodes into many false positives.

## Recommended action plan

1. **P0: add idiom span deduplication / clustering**
   Keep one prediction per matched token span or anchor instead of emitting every neighboring canonical entry. This directly attacks the `duplicate_on_true_mwe` bucket.

2. **P0: gate placeholder-heavy idiom patterns**
   Entries with `someone`, `something`, `some place`, `an animal`, etc. should not be first-class default detections. This is the single biggest bucket.

3. **P1: make idiom matching syntax-aware**
   At minimum: sentence boundaries, tighter token distance, and POS sanity checks. Better: dependency-based validation for the most error-prone families.

4. **P1: split the idiom inventory into confidence tiers**
   Generic entries like `at work`, `more and more`, `go there`, `do what` should be demoted or excluded from default detection.

5. **P2: review the small residue manually**
   `get out of here`, `hang with`, `do better`, and similar cases are small enough to handle with targeted lexicon curation or gold-guideline updates.

If you want to turn this analysis into implementation work, the first code changes I would make are:
- dedupe idiom predictions before writing `pred_rows`
- skip or down-rank placeholder-rich idiom patterns when building the idiom index
- tighten `_find_lemma_sequence()` so punctuation and sentence boundaries matter
